In [ ]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Sandip1000/vatex_train_ckpt",
    filename="checkpoint-8000/model.safetensors"
)

print(file_path)

In [ ]:
! pip install av 

! pip install sacrebleu rouge-score pycocoevalcap evaluate

! pip install bert_score

## Import Liraries and configuration


In [ ]:
import os
import re
import json
import torch
import torch.nn as nn
import av
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from typing import List

from transformers import (
    AutoImageProcessor,
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    TimesformerModel,
    AutoModelForSeq2SeqLM,
    NllbTokenizerFast
)
from transformers.modeling_outputs import BaseModelOutput


In [ ]:
# For model.safetensors
from safetensors.torch import load_file

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  
# ══════════════════════════════════════════════════════════════════════════════
DatasetFolder = fr"/kaggle/input/datasets/sandipsanjel/vatex-nepali/VATEX_ne"
TEST_VIDEOS_DIR   = os.path.join(DatasetFolder, "videos_test")
TEST_CAPTIONS_FILE = os.path.join(DatasetFolder, "nepali_captions/vatex_test_ne.json")
CHECKPOINT_PATH   =fr"/root/.cache/huggingface/hub/models--Sandip1000--vatex_train_ckpt/snapshots/22b90cb9429834efd49be96eaf6fb7dd08d4b9b0/checkpoint-8000/model.safetensors"
OUTPUT_DIR        = fr"/kaggle/working/evaluation_results"

NUM_FRAMES  = 8
MAX_LENGTH  = 32          
NUM_BEAMS   = 4
BATCH_SIZE  = 256

encoder_name    = "facebook/timesformer-base-finetuned-k600"       # must match training
decoder_name    = "facebook/mbart-large-50"                        # must match training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Model - Timesformer frozen + qformer+ mBART

In [ ]:

# ============================================================
# CELL 16 — Q-Former Block WITH dropout
# FIX: added dropout throughout
# ============================================================
class QFormerBlock(nn.Module):
    def __init__(self, hidden_dim=1024, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim   = hidden_dim,
            num_heads   = num_heads,
            batch_first = True,
            dropout     = dropout,        # FIX: added dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)

        self.self_attn = nn.MultiheadAttention(
            embed_dim   = hidden_dim,
            num_heads   = num_heads,
            batch_first = True,
            dropout     = dropout,        # FIX: added dropout
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, int(hidden_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),          # FIX: added dropout in MLP
            nn.Linear(int(hidden_dim * mlp_ratio), hidden_dim),
        )
        self.norm3   = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, video_features):
        cross_out, _ = self.cross_attn(
            query=queries,
            key  =video_features,
            value=video_features,
        )
        queries = self.norm1(queries + self.dropout(cross_out))   # FIX: dropout

        self_out, _ = self.self_attn(
            query=queries,
            key  =queries,
            value=queries,
        )
        queries = self.norm2(queries + self.dropout(self_out))    # FIX: dropout

        mlp_out = self.mlp(queries)
        queries = self.norm3(queries + self.dropout(mlp_out))     # FIX: dropout

        return queries


# ============================================================
# CELL 17 — Video Captioning Model
# FIX: replaced mBART with NLLB-600M, partial TimeSformer unfreeze,
#      frozen decoder embeddings
# ============================================================
class VideoCaptioningModel(nn.Module):
    def __init__(
        self,
        timesformer_model_name = "facebook/timesformer-base-finetuned-k600",
        nllb_model_name        = "facebook/nllb-200-distilled-600M",  # FIX: was mBART
        num_query_tokens       = 50,
        qformer_layers         = 4,
        dropout                = 0.1,
        freeze_timesformer     = True,
        unfreeze_last_n_blocks = 3,           # FIX: partial unfreeze
        freeze_nllb_encoder    = True,
    ):
        super().__init__()

        # -------- Encoder: TimeSformer --------
        self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

        # -------- Decoder: NLLB-600M (FIX: was mBART) --------
        self.decoder = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name)

        ts_hidden   = self.encoder.config.hidden_size   # 768
        nllb_hidden = self.decoder.config.d_model       # 1024

        # -------- Projection (768 → 1024) --------
        self.video_proj = nn.Linear(ts_hidden, nllb_hidden)

        # -------- Q-Former --------
        self.query_tokens = nn.Parameter(
            torch.randn(1, num_query_tokens, nllb_hidden)
        )
        self.qformer_blocks = nn.ModuleList([
            QFormerBlock(hidden_dim=nllb_hidden, dropout=dropout)
            for _ in range(qformer_layers)
        ])

        # -------- Freeze TimeSformer --------
        if freeze_timesformer:
            for p in self.encoder.parameters():
                p.requires_grad = False
        
            # Unfreeze last 3 blocks
            for block in self.encoder.encoder.layer[-unfreeze_last_n_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
        
            print(f"TimeSformer: frozen except last {unfreeze_last_n_blocks} blocks ✅")
        
        # -------- Freeze NLLB --------
        if freeze_nllb_encoder:
            # Freeze entire encoder (not used anyway)
            for p in self.decoder.model.encoder.parameters():
                p.requires_grad = False
        
            # Train decoder layers
            for p in self.decoder.model.decoder.parameters():
                p.requires_grad = True
        
            # # Freeze embeddings (already knows Nepali perfectly)
            # for p in self.decoder.model.shared.parameters():
            #     p.requires_grad = False
        
            # # Freeze lm_head (same matrix as shared embeddings)
            # for p in self.decoder.lm_head.parameters():
            #     p.requires_grad = False
                
            #     # Freeze lm_head (same matrix as shared embeddings)
            # for p in self.decoder.model.decoder.embed_tokens.parameters():
            #     p.requires_grad = False
                
        
            # print("NLLB: encoder frozen, decoder trainable, embeddings frozen ✅")
        
        # -------- Verify --------
        # total     = sum(p.numel() for p in self.parameters())
        # trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        # print(f"Total params:     {total/1e6:.1f}M")
        # print(f"Trainable params: {trainable/1e6:.1f}M")
        # print(f"Frozen params:    {(total-trainable)/1e6:.1f}M")

    # --------------------------------------------------
    # Forward (Training)
    # --------------------------------------------------
    def forward(self, pixel_values, input_ids, labels):

        # 1. TimeSformer Encoding
        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict =True
        )
        video_features = encoder_outputs.last_hidden_state  # (B, N, 768)

        # 2. Project to NLLB hidden size
        video_features = self.video_proj(video_features)    # (B, N, 1024)

        B = video_features.size(0)

        # 3. Expand Query Tokens
        queries = self.query_tokens.expand(B, -1, -1)       # (B, 32, 1024)

        # 4. Q-Former Processing
        for block in self.qformer_blocks:
            queries = block(queries, video_features)         # (B, 32, 1024)

        # 5. Wrap as Encoder Memory for NLLB decoder
        encoder_out = BaseModelOutput(last_hidden_state=queries)
        encoder_attention_mask = torch.ones(
            queries.size()[:-1],
            dtype =torch.long,
            device=queries.device
        )

        # 6. NLLB Decoder forward
        outputs = self.decoder(
            encoder_outputs  =encoder_out,
            attention_mask   =encoder_attention_mask,
            input_ids        =input_ids,
            labels           =labels,
            return_dict      =True
        )

        return outputs

    # --------------------------------------------------
    # Generate (Inference)
    # --------------------------------------------------
    @torch.no_grad()
    def generate(
        self,
        pixel_values,
        max_length           = 50,            # FIX: was 32
        num_beams            = 4,
        forced_bos_token_id  = None,
        **generate_kwargs
    ):
        encoder_outputs = self.encoder(
            pixel_values=pixel_values,
            return_dict =True
        )
        video_features = self.video_proj(encoder_outputs.last_hidden_state)

        B       = video_features.size(0)
        queries = self.query_tokens.expand(B, -1, -1)

        for block in self.qformer_blocks:
            queries = block(queries, video_features)

        encoder_out = BaseModelOutput(last_hidden_state=queries)
        encoder_attention_mask = torch.ones(
            queries.size()[:-1],
            dtype =torch.long,
            device=queries.device
        )

        generated_ids = self.decoder.generate(
            encoder_outputs  =encoder_out,
            attention_mask   =encoder_attention_mask,
            max_length       =max_length,
            num_beams        =num_beams,
            forced_bos_token_id=forced_bos_token_id,
            no_repeat_ngram_size=3,
            early_stopping   =True,
            **generate_kwargs
        )

        return generated_ids





In [ ]:
# # Timesformer + Qformer + mBART

# # --------------------------------------------------
# # Q-Former Block
# # --------------------------------------------------
# class QFormerBlock(nn.Module):
#     def __init__(self, hidden_dim=1024, num_heads=8, mlp_ratio=4.0):
#         super().__init__()

#         # Cross-attention (Queries attend to video features)
#         self.cross_attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim,
#             num_heads=num_heads,
#             batch_first=True,
#         )
#         self.norm1 = nn.LayerNorm(hidden_dim)

#         # Self-attention among queries
#         self.self_attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim,
#             num_heads=num_heads,
#             batch_first=True,
#         )
#         self.norm2 = nn.LayerNorm(hidden_dim)

#         # Feed-forward
#         self.mlp = nn.Sequential(
#             nn.Linear(hidden_dim, int(hidden_dim * mlp_ratio)),
#             nn.GELU(),
#             nn.Linear(int(hidden_dim * mlp_ratio), hidden_dim),
#         )
#         self.norm3 = nn.LayerNorm(hidden_dim)

#     def forward(self, queries, video_features):
#         # Cross-attention
#         cross_out, _ = self.cross_attn(
#             query=queries,
#             key=video_features,
#             value=video_features,
#         )
#         queries = self.norm1(queries + cross_out)

#         # Self-attention
#         self_out, _ = self.self_attn(
#             query=queries,
#             key=queries,
#             value=queries,
#         )
#         queries = self.norm2(queries + self_out)

#         # MLP
#         mlp_out = self.mlp(queries)
#         queries = self.norm3(queries + mlp_out)

#         return queries


# # --------------------------------------------------
# # Video Captioning Model with Q-Former Bridge
# # --------------------------------------------------
# class VideoCaptioningModel(nn.Module):
#     def __init__(
#         self,
#         timesformer_model_name="facebook/timesformer-base-finetuned-k600",
#         mbart_model_name="facebook/mbart-large-50",
#         num_query_tokens=32,
#         qformer_layers=4,
#         freeze_timesformer=True,
#         freeze_mbart_encoder=True,
#     ):
#         super().__init__()

#         # -------- Encoder: TimeSformer --------
#         self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

#         # -------- Decoder: mBART --------
#         self.decoder = MBartForConditionalGeneration.from_pretrained(mbart_model_name)

#         ts_hidden = self.encoder.config.hidden_size        # usually 768
#         mbart_hidden = self.decoder.config.d_model         # 1024

#         # -------- Projection (768 → 1024) --------
#         self.video_proj = nn.Linear(ts_hidden, mbart_hidden)

#         # -------- Q-Former --------
#         self.query_tokens = nn.Parameter(
#             torch.randn(1, num_query_tokens, mbart_hidden)
#         )

#         self.qformer_blocks = nn.ModuleList(
#             [QFormerBlock(hidden_dim=mbart_hidden) for _ in range(qformer_layers)]
#         )

#         # -------- Freeze modules --------
#         if freeze_timesformer:
#             for p in self.encoder.parameters():
#                 p.requires_grad = False

#         if freeze_mbart_encoder:
#             for p in self.decoder.model.encoder.parameters():
#                 p.requires_grad = False
#             for p in self.decoder.model.decoder.parameters():
#                 p.requires_grad = True

#     # --------------------------------------------------
#     # Forward (Training)
#     # --------------------------------------------------
#     def forward(self, pixel_values, input_ids, labels):

#         # 1️⃣ TimeSformer Encoding
#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )

#         video_features = encoder_outputs.last_hidden_state  # (B, N, 768)

#         # 2️⃣ Project to 1024
#         video_features = self.video_proj(video_features)

#         B = video_features.size(0)

#         # 3️⃣ Expand Query Tokens
#         queries = self.query_tokens.expand(B, -1, -1)

#         # 4️⃣ Q-Former Processing
#         for block in self.qformer_blocks:
#             queries = block(queries, video_features)

#         # queries shape: (B, num_query_tokens, 1024)

#         # 5️⃣ Wrap as Encoder Memory for mBART
#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=queries
#         )
        
#         encoder_attention_mask = torch.ones(
#         queries.size()[:-1],
#         dtype=torch.long,
#         device=queries.device
#         )

#         # 6️⃣ mBART Decoder
#         outputs = self.decoder(
#             encoder_outputs=encoder_outputs,
#             attention_mask=encoder_attention_mask,
#             input_ids=input_ids,
#             labels=labels,
#             return_dict=True
#         )

#         return outputs

#     # --------------------------------------------------
#     # Generate (Inference)
#     # --------------------------------------------------
#     @torch.no_grad()
#     def generate(
#         self,
#         pixel_values,
#         max_length=50,
#         num_beams=4,
#         forced_bos_token_id=None,
#         decoder_start_token_id=None,
#         **generate_kwargs
#     ):

#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )

#         video_features = self.video_proj(
#             encoder_outputs.last_hidden_state
#         )

#         B = video_features.size(0)
#         queries = self.query_tokens.expand(B, -1, -1)

#         for block in self.qformer_blocks:
#             queries = block(queries, video_features)

#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=queries
#         )

#         encoder_attention_mask = torch.ones(
#         queries.size()[:-1],
#         dtype=torch.long,
#         device=queries.device
#        ) 

#         generated_ids = self.decoder.generate(
#             encoder_outputs=encoder_outputs,
#             attention_mask=encoder_attention_mask,
#             max_length=max_length,
#             num_beams=num_beams,
#             forced_bos_token_id=forced_bos_token_id,
#             decoder_start_token_id=decoder_start_token_id,
#             **generate_kwargs
#         )

#         return generated_ids

## Model- Timesformer + Mbart Linear Projection


In [ ]:
# # Timesformer + mBART
# class VideoCaptioningModel(nn.Module):
#     def __init__(self,
#         timesformer_model_name="facebook/timesformer-base-finetuned-k600",
#         mbart_model_name="facebook/mbart-large-50",
#         freeze_timesformer=True,
#     ):
#         super().__init__()

#         # -------- Encoder: TimeSformer --------
#         self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

#         # -------- Decoder: mBART (encoder unused) --------
#         self.decoder = MBartForConditionalGeneration.from_pretrained(mbart_model_name)

#         # -------- Dimensions --------
#         ts_hidden = self.encoder.config.hidden_size          # e.g. 768
#         mbart_hidden = self.decoder.config.d_model           # e.g. 1024

#         # -------- Adapter (TimeSformer → mBART) --------
#         self.adapter = nn.Sequential(
#             nn.LayerNorm(ts_hidden),
#             nn.Linear(ts_hidden, mbart_hidden),
#             nn.GELU(),
#             nn.LayerNorm(mbart_hidden)
#         )


#         # Initialize linear layer properly
#         nn.init.normal_(self.adapter[1].weight, mean=0, std=0.02)
#         nn.init.zeros_(self.adapter[1].bias)

        
#         # Freeze TimeSformer
#         if freeze_timesformer:
#             for p in self.encoder.parameters():
#                 p.requires_grad = False
                
#         # Freeze the MBART encoder
#         for p in self.decoder.model.encoder.parameters():
#             p.requires_grad = False

#     def forward(self, pixel_values, input_ids, labels):
#         """
#         pixel_values : video tensor
#         input_ids    : decoder tokens
#         labels       : for training (optional)
#         """

#         # -------- 1. TimeSformer Encoding --------
#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )

#         # (B, seq_len, ts_hidden)
#         video_features = encoder_outputs.last_hidden_state

#         # -------- 2. Adapter Projection --------
#         adapted_features = self.adapter(video_features)
#         # (B, seq_len, mbart_hidden)

#         # -------- 3. Wrap in BaseModelOutput --------
#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=adapted_features
#         )

#         # -------- 4. mBART Decoder with Cross-Attention --------
#         outputs = self.decoder(
#             encoder_outputs = encoder_outputs,  # <-- cross-attention source
#             input_ids = input_ids,
#             labels = labels,
#             return_dict = True
#         )

#         return outputs

#     @torch.no_grad()
#     def generate(
#         self,
#         pixel_values,             # video input
#         max_length=50,
#         num_beams=4,
#         forced_bos_token_id=None,
#         decoder_start_token_id=None,
#         **generate_kwargs
#     ):
#         """
#         Generates captions for given video frames.
#         """
#         self.eval()

#         # 1. TimeSformer Encoding
#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )
#         video_features = encoder_outputs.last_hidden_state

#         # 2. Adapter projection
#         adapted_features = self.adapter(video_features)

#         # 3. Wrap in BaseModelOutput
#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=adapted_features
#         )

#         # 4. Generate with mBART
#         generated_ids = self.decoder.generate(
#             encoder_outputs=encoder_outputs,
#             max_length=max_length,
#             num_beams=num_beams,
#             forced_bos_token_id=forced_bos_token_id,
#             decoder_start_token_id=decoder_start_token_id,
#             **generate_kwargs
#         )

#         return generated_ids

## Model - Timesformer Unfreeze + Qformer + mBART


In [ ]:
# class QFormerBlock(nn.Module):
#     def __init__(self, hidden_dim=1024, num_heads=8, mlp_ratio=4.0, dropout=0.1):
#         super().__init__()

#         # Cross-attention (Queries attend to video features)
#         self.cross_attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim,
#             num_heads=num_heads,
#             batch_first=True,
#             dropout=dropout  # dropout inside attention
#         )
#         self.norm1 = nn.LayerNorm(hidden_dim)
#         self.cross_dropout = nn.Dropout(dropout)  # after residual

#         # Self-attention among queries
#         self.self_attn = nn.MultiheadAttention(
#             embed_dim=hidden_dim,
#             num_heads=num_heads,
#             batch_first=True,
#             dropout=dropout
#         )
#         self.norm2 = nn.LayerNorm(hidden_dim)
#         self.self_dropout = nn.Dropout(dropout)

#         # Feed-forward
#         self.mlp = nn.Sequential(
#             nn.Linear(hidden_dim, int(hidden_dim * mlp_ratio)),
#             nn.GELU(),
#             nn.Dropout(dropout),  # MLP dropout
#             nn.Linear(int(hidden_dim * mlp_ratio), hidden_dim),
#         )
#         self.norm3 = nn.LayerNorm(hidden_dim)
#         self.mlp_dropout = nn.Dropout(dropout)

#     def forward(self, queries, video_features):
#         # Cross-attention
#         cross_out, _ = self.cross_attn(
#             query=queries,
#             key=video_features,
#             value=video_features,
#         )
#         queries = self.norm1(queries + self.cross_dropout(cross_out))

#         # Self-attention
#         self_out, _ = self.self_attn(
#             query=queries,
#             key=queries,
#             value=queries,
#         )
#         queries = self.norm2(queries + self.self_dropout(self_out))

#         # MLP
#         mlp_out = self.mlp(queries)
#         queries = self.norm3(queries + self.mlp_dropout(mlp_out))

#         return queries

# # --------------------------------------------------
# # Video Captioning Model with Q-Former Bridge
# # --------------------------------------------------
# class VideoCaptioningModel(nn.Module):
#     def __init__(
#         self,
#         timesformer_model_name="facebook/timesformer-base-finetuned-k600",
#         mbart_model_name="facebook/mbart-large-50",
#         num_query_tokens=32,
#         qformer_layers=4,
#         freeze_timesformer=True,
#         freeze_mbart_encoder=True,
#     ):
#         super().__init__()

#         # -------- Encoder: TimeSformer --------
#         self.encoder = TimesformerModel.from_pretrained(timesformer_model_name)

#         # -------- Decoder: mBART --------
#         self.decoder = MBartForConditionalGeneration.from_pretrained(mbart_model_name)

#         ts_hidden = self.encoder.config.hidden_size        # usually 768
#         mbart_hidden = self.decoder.config.d_model         # 1024

#         # -------- Projection (768 → 1024) --------
#         self.video_proj = nn.Linear(ts_hidden, mbart_hidden)

#         # -------- Q-Former --------
#         self.query_tokens = nn.Parameter(
#             torch.randn(1, num_query_tokens, mbart_hidden)
#         )

#         self.qformer_blocks = nn.ModuleList(
#             [QFormerBlock(hidden_dim=mbart_hidden) for _ in range(qformer_layers)]
#         )

#         # Qformer_total_parameters = sum(p.numel() for p in self.qformer_blocks.parameters())
#         # Qformer_trainable_parameters = sum(p.numel() for p in self.qformer_blocks.parameters() if p.requires_grad)
#         # print("Qformer Total parameters: ", Qformer_total_parameters)
#         # print("Qformer Trainable parameters: ", Qformer_trainable_parameters)
#         # print("Qformer Non-Trainable parameters: ", Qformer_total_parameters- Qformer_trainable_parameters)

#         # -------- Freeze modules --------
#         if freeze_timesformer:
#             for p in self.encoder.parameters():
#                 p.requires_grad = False
        
#         #Unfreeze last 4 TimeSformer blocks
#         for block in self.encoder.encoder.layer[-4:]:
#             for p in block.parameters():
#                 p.requires_grad = True
        
#         # timesformer_total_parameters = sum(p.numel() for p in self.encoder.parameters())
#         # timesformer_trainable_parameters = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
#         # print("Timesformer Total parameters: ", timesformer_total_parameters)
#         # print("Timesformer Trainable parameters: ", timesformer_trainable_parameters)
#         # print("Timesformer Non-Trainable parameters: ", timesformer_total_parameters- timesformer_trainable_parameters)


        
#         if freeze_mbart_encoder:
#             for p in self.decoder.model.encoder.parameters():
#                 p.requires_grad = False
#             for p in self.decoder.model.decoder.parameters():
#                 p.requires_grad = True

        
#         # mbart_total_parameters = sum(p.numel() for p in self.decoder.parameters())
#         # mbart_trainable_parameters = sum(p.numel() for p in self.decoder.parameters() if p.requires_grad)
#         # print("mBART Total parameters: ", mbart_total_parameters)
#         # print("mBART Trainable parameters: ", mbart_trainable_parameters)
#         # print("mBART Non-Trainable parameters: ", mbart_total_parameters- mbart_trainable_parameters)

#     # --------------------------------------------------
#     # Forward (Training)
#     # --------------------------------------------------
#     def forward(self, pixel_values, input_ids, labels):

#         # 1️⃣ TimeSformer Encoding
#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )

#         video_features = encoder_outputs.last_hidden_state  # (B, N, 768)

#         # 2️⃣ Project to 1024
#         video_features = self.video_proj(video_features)

#         B = video_features.size(0)

#         # 3️⃣ Expand Query Tokens
#         queries = self.query_tokens.expand(B, -1, -1)

#         # 4️⃣ Q-Former Processing
#         for block in self.qformer_blocks:
#             queries = block(queries, video_features)

#         # queries shape: (B, num_query_tokens, 1024)

#         # 5️⃣ Wrap as Encoder Memory for mBART
#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=queries
#         )
        
#         encoder_attention_mask = torch.ones(
#         queries.size()[:-1],
#         dtype=torch.long,
#         device=queries.device
#         )

#         # 6️⃣ mBART Decoder
#         outputs = self.decoder(
#             encoder_outputs=encoder_outputs,
#             attention_mask=encoder_attention_mask,
#             input_ids=input_ids,
#             labels=labels,
#             return_dict=True
#         )

#         return outputs

#     # --------------------------------------------------
#     # Generate (Inference)
#     # --------------------------------------------------
#     @torch.no_grad()
#     def generate(
#         self,
#         pixel_values,
#         max_length=32,
#         num_beams=4,
#         forced_bos_token_id=None,
#         decoder_start_token_id=None,
#         **generate_kwargs
#     ):

#         encoder_outputs = self.encoder(
#             pixel_values=pixel_values,
#             return_dict=True
#         )

#         video_features = self.video_proj(
#             encoder_outputs.last_hidden_state
#         )

#         B = video_features.size(0)
#         queries = self.query_tokens.expand(B, -1, -1)

#         for block in self.qformer_blocks:
#             queries = block(queries, video_features)

#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=queries
#         )

#         encoder_attention_mask = torch.ones(
#         queries.size()[:-1],
#         dtype=torch.long,
#         device=queries.device
#        ) 

#         generated_ids = self.decoder.generate(
#             encoder_outputs=encoder_outputs,
#             attention_mask=encoder_attention_mask,
#             max_length=max_length,
#             num_beams=num_beams,
#             forced_bos_token_id=forced_bos_token_id,
#             decoder_start_token_id=decoder_start_token_id,
#             **generate_kwargs
#         )

#         return generated_ids

## Processing and Infrencing


In [ ]:
# ═════════════════════
# VIDEO PREPROCESSING 
# ═════════════════════
def read_video_pyav(video_path: str, num_frames: int = NUM_FRAMES):
    """Decode all frames with PyAV then sample uniformly — same as training."""
    container = av.open(video_path)
    frames = []
    for frame in container.decode(video=0):
        frames.append(frame.to_rgb().to_ndarray())
    container.close()

    idx    = np.linspace(0, max(len(frames) - 1, 0), num_frames).astype(np.int64)
    frames = [frames[i] for i in idx]
    return frames  # list of (H, W, C) numpy arrays


def process_video(video_path: str, image_processor) -> torch.Tensor:
    """Returns pixel_values tensor of shape (1, T, C, H, W) — same as training."""
    frames = read_video_pyav(video_path)
    inputs = image_processor(list(frames), return_tensors="pt")
    return inputs["pixel_values"]   # (1, T, C, H, W)



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════
def load_test_data(captions_file: str, videos_dir: str):
    """
    Returns one entry per UNIQUE VIDEO (not per caption).
    Each entry: {"video_id": str, "video_path": str, "references": [str, ...]}
    """
    with open(captions_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # handle both {"root": [...]} and plain [...]
    if isinstance(data, dict) and "root" in data:
        data = data["root"]

    samples, missing = [], []
    for item in data:
        video_id   = item["video_id"]
        references = item["caption"]
        video_path = os.path.join(videos_dir, f"{video_id}.mp4")
        if not os.path.exists(video_path):
            missing.append(video_id)
            continue
        samples.append({"video_id": video_id, "video_path": video_path, "references": references})

    print(f"Loaded {len(samples)} test videos")
    if missing:
        print(f"  Missing {len(missing)} videos: {missing[:5]}{'...' if len(missing)>5 else ''}")
    return samples

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INFERENCE
# ══════════════════════════════════════════════════════════════════════════════
@torch.no_grad()
def generate_predictions(model, tokenizer, image_processor, test_samples):
    """
    Returns:
        predictions : list[str]          – one caption per video (best beam)
        references  : list[list[str]]    – all reference captions per video
        video_ids   : list[str]
    """
    model.eval()
    predictions, references, video_ids = [], [], []
    failed = []

    ne_lang_id = tokenizer.convert_tokens_to_ids("ne_NP")

    for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Inference"):
        batch = test_samples[i : i + BATCH_SIZE]
        batch_tensors, batch_refs, batch_ids = [], [], []

        for sample in batch:
            try:
                # process_video returns (1, T, C, H, W) — squeeze the batch dim
                tensor = process_video(sample["video_path"], image_processor)
                tensor = tensor.squeeze(0)          # (T, C, H, W)
                batch_tensors.append(tensor)
                batch_refs.append(sample["references"])
                batch_ids.append(sample["video_id"])
            except Exception as e:
                print(f"\n  Error on {sample['video_id']}: {e}")
                failed.append(sample["video_id"])

        if not batch_tensors:
            continue

        pixel_values = torch.stack(batch_tensors).to(device)  # (B, T, C, H, W)

        generated_ids = model.generate(
            pixel_values=pixel_values,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            num_return_sequences=NUM_BEAMS,   # return all beams for inspection
            forced_bos_token_id=ne_lang_id,
            no_repeat_ngram_size=3,
            early_stopping=True,
            repetition_penalty=1.3
        )

        # ── decode all beams ──────────────────────────────────────────────────
        # generated_ids shape: (B * NUM_BEAMS, seq_len)
        # layout: [vid0_beam0, vid0_beam1, ..., vid1_beam0, vid1_beam1, ...]
        all_captions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        # ── debug: print all beams for first batch only ───────────────────────
        if i == 0:
            print("\n" + "=" * 60)
            print("  ALL BEAMS — first video in first batch")
            print("=" * 60)
            for j in range(NUM_BEAMS):
                print(f"  beam {j} : {all_captions[j]}")
            print(f"\n  reference 1 : {batch_refs[0][0]}")
            print(f"  reference 2 : {batch_refs[0][1] if len(batch_refs[0]) > 1 else 'N/A'}")
            print("=" * 60 + "\n")

        # ── keep only beam 0 (best beam) per video ────────────────────────────
        # beam 0 for video j is at index j * NUM_BEAMS
        best_captions = [all_captions[j * NUM_BEAMS] for j in range(len(batch_ids))]

        predictions.extend(best_captions)
        references.extend(batch_refs)
        video_ids.extend(batch_ids)

    print(f"\nGenerated {len(predictions)} predictions")
    if failed:
        print(f"  Failed videos : {len(failed)}")
        for vid in failed:
            print(f"    - {vid}")

    return predictions, references, video_ids

## Metrics

In [ ]:
"""
Evaluation metrics for Nepali Video Captioning.

Metrics computed:
    - BLEU-1/2/3/4  : sacrebleu corpus_score (multi-ref, correct clipping)
    - ROUGE-L       : word-level LCS, β=1.2 (captioning standard),
                      max precision + max recall across all refs → F1
    - METEOR        : HuggingFace evaluate, all refs passed together
                      (metric handles best-match internally)
    - ChrF++        : sacrebleu corpus_score (multi-ref)
    - BERTScore-F1  : xlm-roberta-large, max F1 over all refs,
                      rescaled with baseline for human-interpretable scores
    - CIDEr         : pycocoevalcap Cider (CIDEr-D unavailable in pip v1.2)

Design decisions and justifications:
──────────────────────────────────────────────────────────────────────────────
1. TOKENIZER — Devanagari-aware punctuation splitting
   Naive str.split() attaches Purna Biram (।) and other punctuation to the
   preceding word: "छ।" ≠ "छ" even when semantically identical. This
   unfairly penalizes BLEU, ROUGE-L, and CIDEr. We use regex to separate
   all punctuation from words before scoring, following Indic NLP convention.

2. ROUGE-L β=1.2 — captioning benchmark standard
   The pycocoevalcap implementation (used by MS-COCO, VATEX, MSR-VTT) uses
   β=1.2 which weights recall higher than precision. Using β=1.0 produces
   scores systematically lower than published benchmarks, making cross-paper
   comparison invalid. β=1.2 is required for reproducible comparison.

3. ROUGE-L aggregation — max precision + max recall separately
   pycocoevalcap standard: find max precision across all refs, find max
   recall across all refs independently, then compute F1 from those maxima.
   This is more lenient than max(F1) and aligns with the benchmark standard.

4. BERTSCORE AGGREGATION — max over all refs
   The official bert_score package standard for multi-reference evaluation
   selects the maximum F1 per candidate-reference pair. This aligns with
   the best-match policy used across all captioning benchmarks.

5. CIDEr-D — defensive variant
   CIDEr-D adds length penalties and clipping to prevent models from gaming
   scores with long repetitive captions. MSR-VTT and most leaderboards use
   CIDEr-D as the primary ranking metric, not plain CIDEr.

6. METEOR — exact-match only (limitation acknowledged)
   The HuggingFace METEOR library has no Nepali stemmer. It falls back to
   exact token matching, making it essentially a weighted unigram BLEU.
   Morphological variants like "पकाउँदै" and "पकाउने" will NOT be matched.
   This is noted explicitly in the results output.

7. BERTScore — rescaled with baseline
   Raw xlm-roberta-large cosine similarities cluster between 0.85–0.95
   even for unrelated sentences, making raw scores uninterpretable.
   rescale_with_baseline=True maps scores to [0, 1] where 0 = no better
   than random and 1 = perfect match. This is the research standard.
"""

import re
import math
from typing import List, Dict, Tuple

from tqdm import tqdm
import sacrebleu as scb
import evaluate
from pycocoevalcap.cider.cider import Cider


# ──────────────────────────────────────────────────────────────────────────────
# Tokenizer
# ──────────────────────────────────────────────────────────────────────────────

# Punctuation characters that must be split from surrounding Devanagari words.
# Purna Biram (।) is the Nepali full stop and almost always touches the last
# word without a space, e.g. "छ।" — naive split() treats this as one token,
# causing "छ।" ≠ "छ" mismatches even when prediction is 100% correct.
_DEVANAGARI_PUNCT = re.compile(r"([।॥,.!?;:\"'()\[\]{}])")


def tokenize_nepali(text: str) -> List[str]:
    """
    Devanagari-aware word tokenizer for Nepali text.

    Steps:
        1. Insert spaces around all punctuation (including Purna Biram ।)
           so that "छ।" becomes "छ ।" before splitting.
        2. Collapse multiple spaces.
        3. Split on whitespace and filter empty tokens.

    Example:
        "भात पकाउँदै छ।"  →  ["भात", "पकाउँदै", "छ", "।"]
        "भात पकाउँदै छ"   →  ["भात", "पकाउँदै", "छ"]
        Both yield the same content tokens, so BLEU/ROUGE-L no longer
        penalizes a prediction for missing a trailing Purna Biram.
    """
    text = _DEVANAGARI_PUNCT.sub(r" \1 ", text)   # separate punctuation
    text = re.sub(r"\s+", " ", text).strip()        # collapse spaces
    return [t for t in text.split() if t]


# ──────────────────────────────────────────────────────────────────────────────
# ROUGE-L (word-level LCS, β=1.2, pycocoevalcap standard)
# ──────────────────────────────────────────────────────────────────────────────

# β=1.2 is the captioning benchmark standard used by pycocoevalcap (MS-COCO,
# VATEX, MSR-VTT). It weights recall more heavily than precision, reflecting
# the importance of content coverage in captioning tasks. Using β=1.0 produces
# systematically lower scores incomparable to published benchmarks.
_ROUGE_BETA = 1.2


def _rouge_l_pr(pred: str, ref: str):
    """
    Compute word-level ROUGE-L precision and recall separately via LCS.

    Returns (precision, recall) tuple.
    Returns (0.0, 0.0) if either string is empty after tokenization.
    """
    pt = tokenize_nepali(pred)
    rt = tokenize_nepali(ref)
    if not pt or not rt:
        return 0.0, 0.0

    m, n = len(pt), len(rt)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pt[i - 1] == rt[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs       = dp[m][n]
    precision = lcs / m
    recall    = lcs / n
    return precision, recall


def compute_rouge_l(predictions: List[str], references: List[List[str]]) -> float:
    """
    Compute corpus-level ROUGE-L aligned with the pycocoevalcap standard.

    Aggregation strategy (pycocoevalcap / MS-COCO / VATEX standard):
        For each video, compute LCS precision and recall against every
        reference independently. Take the MAXIMUM precision across all refs
        and the MAXIMUM recall across all refs separately, then compute
        the final F-score from those two maxima.

        This differs from simply taking max(F1) — it finds the best
        possible precision from any ref AND the best possible recall from
        any ref, which is the most lenient and benchmark-consistent approach.

    β=1.2 (pycocoevalcap default):
        Weights recall more than precision. Required for cross-paper
        comparability with MS-COCO, VATEX, and MSR-VTT leaderboards.

    Args:
        predictions : list of predicted captions (one per video)
        references  : list of reference lists (multiple refs per video)

    Returns:
        Mean of per-video ROUGE-L F-scores * 100
    """
    b2 = _ROUGE_BETA ** 2
    scores = []
    for pred, refs in zip(predictions, references):
        pr_pairs  = [_rouge_l_pr(pred, ref) for ref in refs]
        max_p     = max(p for p, r in pr_pairs)
        max_r     = max(r for p, r in pr_pairs)
        if max_p + max_r == 0:
            scores.append(0.0)
        else:
            # Fβ = (1 + β²) * P * R / (R + β² * P)
            f = ((1 + b2) * max_p * max_r) / (max_r + b2 * max_p)
            scores.append(f)
    return (sum(scores) / len(scores)) * 100


# ──────────────────────────────────────────────────────────────────────────────
# BLEU
# ──────────────────────────────────────────────────────────────────────────────

def compute_bleu(
    predictions: List[str],
    references: List[List[str]]
) -> Dict[str, float]:
    """
    Compute BLEU-1/2/3/4 using sacrebleu corpus_score.

    sacrebleu handles multiple references correctly via modified n-gram
    precision clipping — an n-gram matched against one ref cannot be
    double-counted. No max() issue here.

    Refs are padded with "" and transposed because sacrebleu expects
    refs_transposed[i] = all videos' i-th reference.
    sacrebleu ignores empty string references.

    Args:
        predictions : list of predicted captions
        references  : list of reference lists

    Returns:
        Dict with keys BLEU-1, BLEU-2, BLEU-3, BLEU-4
    """
    max_refs        = max(len(rs) for rs in references)
    padded_r        = [rs + [""] * (max_refs - len(rs)) for rs in references]
    refs_transposed = list(zip(*padded_r))

    bleu4_res = scb.BLEU(max_ngram_order=4).corpus_score(predictions, refs_transposed)
    bp        = bleu4_res.bp
    prec      = bleu4_res.precisions

    def _cumulative_bleu(precisions: List[float], n: int, bp: float) -> float:
        if any(p == 0 for p in precisions[:n]):
            return 0.0
        log_avg = sum(math.log(p / 100) for p in precisions[:n]) / n
        return bp * math.exp(log_avg) * 100

    return {
        "BLEU-1": _cumulative_bleu(prec, 1, bp),
        "BLEU-2": _cumulative_bleu(prec, 2, bp),
        "BLEU-3": _cumulative_bleu(prec, 3, bp),
        "BLEU-4": bleu4_res.score,
        "_refs_transposed": refs_transposed,   # reused by ChrF++
    }


# ──────────────────────────────────────────────────────────────────────────────
# METEOR
# ──────────────────────────────────────────────────────────────────────────────

def compute_meteor(
    predictions: List[str],
    references: List[List[str]]
) -> float:
    """
    Compute corpus-level METEOR for Nepali video captioning.

    Correct usage: pass ALL references together to the metric at once.
    HuggingFace METEOR accepts references as a list-of-lists and handles
    multi-reference matching internally — no manual max() needed.

    ⚠️  Known limitation — No Nepali stemmer:
        The HuggingFace `evaluate` METEOR implementation has no stemmer
        or synonym module for Nepali. It falls back to exact token matching,
        making it behave like a weighted unigram BLEU rather than true METEOR.
        Morphological variants like "पकाउँदै" and "पकाउने" will NOT be matched
        even though they share the same root. Scores should be interpreted
        with this limitation in mind and noted explicitly in any paper.

    Args:
        predictions : list of predicted captions
        references  : list of reference lists

    Returns:
        Mean METEOR score * 100
    """
    meteor = evaluate.load("meteor")

    scores = []
    for pred, refs in tqdm(
        zip(predictions, references),
        total=len(predictions),
        desc="  METEOR"
    ):
        # Pass ALL refs at once — HuggingFace METEOR handles multi-ref natively
        # references must be a list-of-lists: [[ref1, ref2, ...]]
        result = meteor.compute(predictions=[pred], references=[refs])
        scores.append(result["meteor"])

    return (sum(scores) / len(scores)) * 100


# ──────────────────────────────────────────────────────────────────────────────
# ChrF++
# ──────────────────────────────────────────────────────────────────────────────

def compute_chrf(
    predictions: List[str],
    refs_transposed: List[Tuple[str, ...]]
) -> float:
    """
    Compute ChrF++ (word_order=2) using sacrebleu corpus_score.

    sacrebleu handles multiple references correctly. Reuses the
    refs_transposed already computed for BLEU.

    Args:
        predictions     : list of predicted captions
        refs_transposed : transposed reference list from compute_bleu()

    Returns:
        ChrF++ score (already 0-100 scale from sacrebleu)
    """
    return scb.CHRF(word_order=2).corpus_score(predictions, refs_transposed).score


# ──────────────────────────────────────────────────────────────────────────────
# BERTScore
# ──────────────────────────────────────────────────────────────────────────────

def compute_bertscore(
    predictions: List[str],
    references: List[List[str]]
) -> float:
    """
    Compute BERTScore-F1 using xlm-roberta-large with baseline rescaling.

    Model choice — xlm-roberta-large:
        Strong Devanagari coverage. Recommended for non-English evaluation
        in the original BERTScore paper (Zhang et al., 2020).

    Aggregation — max F1 (official bert_score multi-ref standard):
        The official bert_score package handles multiple references by
        selecting the maximum F1 per candidate-reference pair, consistent
        with the best-match policy used across all visual captioning
        benchmarks (MS-COCO, VATEX, MSR-VTT).

    Rescaling — rescale_with_baseline=True:
        Raw xlm-roberta-large cosine similarities naturally cluster between
        0.85 and 0.95 even for completely unrelated Nepali sentences. This
        makes raw scores uninterpretable — a score of 0.90 could mean anything.
        Baseline rescaling (Zhang et al.) maps scores so that:
            0.0 = no better than a random baseline
            1.0 = perfect semantic match
        This is the standard required for publication-quality reporting.
        lang="ne" loads the Nepali-specific baseline statistics.

    Args:
        predictions : list of predicted captions
        references  : list of reference lists

    Returns:
        Rescaled max BERTScore-F1 * 100, or 0.0 on failure
    """
    bert = evaluate.load("bertscore")
    f1_per_video = []

    for pred, refs in tqdm(
        zip(predictions, references),
        total=len(predictions),
        desc="  BERTScore"
    ):
        res = bert.compute(
            predictions=[pred] * len(refs),
            references=refs,
            model_type="xlm-roberta-large",
            rescale_with_baseline=True,   # makes scores human-interpretable
            lang="ne",                    # Nepali baseline statistics
        )
        # max F1 over all refs — official bert_score multi-ref standard
        f1_per_video.append(max(res["f1"]))

    return (sum(f1_per_video) / len(f1_per_video)) * 100


# ──────────────────────────────────────────────────────────────────────────────
# CIDEr
# ──────────────────────────────────────────────────────────────────────────────

def compute_cider(
    predictions: List[str],
    references: List[List[str]]
) -> float:
    """
    Compute CIDEr using pycocoevalcap.

    Note on CIDEr-D:
        CIDEr-D (Defensive) is the preferred variant for leaderboards like
        MSR-VTT and VATEX as it adds length penalties and clipping. However,
        pycocoevalcap (v1.2, the only published pip version) does NOT include
        a ciderd module. We use plain CIDEr which is the only option available
        from the standard package. This should be noted in any paper.

    CIDEr uses all references for TF-IDF weighting — no max/mean issue.
    It rewards predictions aligned with the human consensus while penalizing
    uninformative common words.

    Args:
        predictions : list of predicted captions
        references  : list of reference lists

    Returns:
        CIDEr score * 100, or 0.0 on failure
    """
    gts = {i: rs             for i, rs in enumerate(references)}
    res = {i: [predictions[i]] for i in range(len(predictions))}
    cider_score, _ = Cider().compute_score(gts, res)
    return cider_score * 100

In [ ]:

def compute_all_metrics(
    predictions: List[str],
    refs_nested: List[List[str]]
) -> Dict[str, float]:
    """
    Compute all evaluation metrics for Nepali video captioning.

    Args:
        predictions : list of model-generated captions (one per video)
        refs_nested : list of human reference lists (multiple per video)

    Returns:
        Dict of metric name → score
    """
    # ── normalise — tokenize then rejoin so all metrics see clean text ────────
    # We tokenize with the Devanagari-aware tokenizer (splits punctuation) then
    # rejoin with spaces. This ensures "छ।" → "छ ।" consistently across all
    # metrics including BLEU, ROUGE-L, CIDEr, and METEOR.
    def _normalise(text: str) -> str:
        return " ".join(tokenize_nepali(text))

    preds_norm = [_normalise(p) for p in predictions]
    refs_norm  = [
        [_normalise(r) for r in rs if r.strip()]
        for rs in refs_nested
    ]

    # ── drop empty pairs ──────────────────────────────────────────────────────
    clean_p, clean_r = [], []
    for p, rs in zip(preds_norm, refs_norm):
        if p and rs:
            clean_p.append(p)
            clean_r.append(rs)

    if not clean_p:
        print("⚠️  No valid prediction-reference pairs found.")
        return {}

    n_videos = len(clean_p)
    n_refs   = sum(len(rs) for rs in clean_r)
    print(f"\nEvaluating {n_videos} videos | {n_refs} total references "
          f"| avg {n_refs / n_videos:.1f} refs/video\n")

    results: Dict[str, float] = {}

    # ── BLEU ─────────────────────────────────────────────────────────────────
    print("Computing BLEU ...")
    bleu_results = compute_bleu(clean_p, clean_r)
    refs_transposed = bleu_results.pop("_refs_transposed")   # reuse for ChrF++
    results.update(bleu_results)
    print(f"  BLEU-1: {results['BLEU-1']:.2f} | BLEU-4: {results['BLEU-4']:.2f}")

    # ── ROUGE-L ───────────────────────────────────────────────────────────────
    print("Computing ROUGE-L (β=1.2, max-P + max-R, pycocoevalcap standard) ...")
    results["ROUGE-L"] = compute_rouge_l(clean_p, clean_r)
    print(f"  ROUGE-L: {results['ROUGE-L']:.2f}")

    # ── METEOR ────────────────────────────────────────────────────────────────
    print("Computing METEOR (all refs passed together) ...")
    results["METEOR"] = compute_meteor(clean_p, clean_r)
    print(f"  METEOR: {results['METEOR']:.2f}")

    # ── ChrF++ ────────────────────────────────────────────────────────────────
    print("Computing ChrF++ ...")
    results["ChrF++"] = compute_chrf(clean_p, refs_transposed)
    print(f"  ChrF++: {results['ChrF++']:.2f}")

    # ── BERTScore ─────────────────────────────────────────────────────────────
    print("Computing BERTScore (max F1, rescaled, lang=ne) ...")
    try:
        results["BERTScore-F1"] = compute_bertscore(clean_p, clean_r)
    except Exception as e:
        print(f"  ⚠️  BERTScore failed: {e}")
        results["BERTScore-F1"] = 0.0
    print(f"  BERTScore-F1: {results['BERTScore-F1']:.2f}")

    # ── CIDEr ─────────────────────────────────────────────────────────────────
    print("Computing CIDEr ...")
    try:
        results["CIDEr"] = compute_cider(clean_p, clean_r)
    except Exception as e:
        print(f"  ⚠️  CIDEr failed: {e}")
        results["CIDEr"] = 0.0
    print(f"  CIDEr: {results['CIDEr']:.2f}")

    # ── summary ───────────────────────────────────────────────────────────────
    results["num_videos"] = n_videos
    results["num_refs"]   = n_refs

    _print_results(results)
    return results


def _print_results(results: Dict[str, float]) -> None:
    """Pretty-print the evaluation results table."""
    n_videos = results["num_videos"]
    n_refs   = results["num_refs"]

    print("\n" + "=" * 60)
    print("  NEPALI VIDEO CAPTIONING — EVALUATION RESULTS")
    print("=" * 60)
    print(f"  Videos evaluated : {n_videos}")
    print(f"  Total references : {n_refs}")
    print(f"  Avg refs / video : {n_refs / n_videos:.1f}")
    print("-" * 60)
    print(f"  BLEU-1           : {results['BLEU-1']:>8.2f}")
    print(f"  BLEU-2           : {results['BLEU-2']:>8.2f}")
    print(f"  BLEU-3           : {results['BLEU-3']:>8.2f}")
    print(f"  BLEU-4           : {results['BLEU-4']:>8.2f}")
    print(f"  ROUGE-L          : {results['ROUGE-L']:>8.2f}  (β=1.2, max-P + max-R, pycocoevalcap)")
    print(f"  METEOR           : {results['METEOR']:>8.2f}  (exact-match only — no Nepali stemmer)")
    print(f"  ChrF++           : {results['ChrF++']:>8.2f}")
    print(f"  BERTScore-F1     : {results['BERTScore-F1']:>8.2f}  (rescaled, max F1, lang=ne)")
    print(f"  CIDEr            : {results['CIDEr']:>8.2f}  (plain CIDEr — CIDEr-D unavailable in pip)")
    print("=" * 60)
    print("\n  METHODOLOGICAL NOTES")
    print("-" * 60)
    print("  Tokenizer  : Devanagari-aware — punctuation (।) split from words")
    print("  ROUGE-L    : β=1.2 (captioning standard), max-P and max-R")
    print("               across all refs separately → F-score")
    print("               (pycocoevalcap / MS-COCO / VATEX standard)")
    print("  METEOR     : ⚠ no Nepali stemmer — treats morphological variants")
    print("               as different words (e.g. पकाउँदै ≠ पकाउने)")
    print("               Interpret as weighted unigram overlap, not true METEOR")
    print("  BERTScore  : max F1 over all refs (official bert_score standard)")
    print("               rescaled with Nepali baseline (lang=ne)")
    print("               0 = random baseline  |  100 = perfect match")
    print("  CIDEr      : plain CIDEr (pycocoevalcap v1.2 — CIDEr-D module")
    print("               not available in pip package, note in paper)")
    print("=" * 60)

In [ ]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ── tokenizer ─────────────────────────────────────────────────────────────
    print("Loading tokenizer ...")
    tokenizer =NllbTokenizerFast.from_pretrained("facebook/nllb-200-distilled-600M", src_lang = "npi_Deva", tgt_lang = "npi_Deva")

    # ── image processor ───────────────────────────────────────────────────────
    print("Loading image processor ...")
    image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")

    # ── model ─────────────────────────────────────────────────────────────────
    print(f"Loading model from {CHECKPOINT_PATH} ...")
    model = VideoCaptioningModel()
    model.decoder.config.tie_word_embeddings = False   # silence tied weights warning

    # ckpt_file = os.path.join(CHECKPOINT_PATH, "model.safetensors")
    # ckpt_file = os.path.join(CHECKPOINT_PATH, "pytorch_model.bin")
    ckpt_file = CHECKPOINT_PATH
    if os.path.exists(ckpt_file):
        state_dict = load_file(ckpt_file, device=str(device))    
        # state_dict = torch.load(ckpt_file, map_location= device)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing:
            print(f"  Missing keys   : {missing[:5]}")
        if unexpected:
            print(f"  Unexpected keys: {unexpected[:5]}")
        print("  ✅ Checkpoint loaded.")
    else:
        print(f"  ⚠️ No checkpoint found at {ckpt_file}. Using untrained model!")

    model.to(device).eval()

    # ── data ──────────────────────────────────────────────────────────────────
    print("Loading test data ...")
    test_samples = load_test_data(TEST_CAPTIONS_FILE, TEST_VIDEOS_DIR)
    if not test_samples:
        print("No test samples found. Exiting.")
        return

    # ── inference ─────────────────────────────────────────────────────────────
    predictions, references, video_ids = generate_predictions(model, tokenizer, image_processor, test_samples)

    if not predictions:
        print("No predictions generated. Exiting.")
        return

    # ── metrics ───────────────────────────────────────────────────────────────
    results = compute_all_metrics(predictions, references)

    # ── save ──────────────────────────────────────────────────────────────────
    with open(os.path.join(OUTPUT_DIR, "results.txt"), "w", encoding="utf-8") as f:

        # ── metrics ───────────────────────────────────────────────────────────
        f.write("=" * 60 + "\n")
        f.write("  EVALUATION RESULTS\n")
        f.write("=" * 60 + "\n")
        f.write(f"  Videos evaluated : {results['num_videos']}\n")
        f.write(f"  Total references : {results['num_refs']}\n")
        f.write(f"  Avg refs/video   : {results['num_refs']/results['num_videos']:.1f}\n")
        f.write("-" * 60 + "\n")
        f.write(f"  BLEU-1 (w/ BP)   : {results['BLEU-1']:>8.2f}\n")
        f.write(f"  BLEU-2 (w/ BP)   : {results['BLEU-2']:>8.2f}\n")
        f.write(f"  BLEU-3 (w/ BP)   : {results['BLEU-3']:>8.2f}\n")
        f.write(f"  BLEU-4 (w/ BP)   : {results['BLEU-4']:>8.2f}\n")
        f.write(f"  ROUGE-L          : {results['ROUGE-L']:>8.2f}\n")
        f.write(f"  METEOR           : {results['METEOR']:>8.2f}\n")
        f.write(f"  ChrF++           : {results['ChrF++']:>8.2f}\n")
        f.write(f"  BERTScore-F1     : {results['BERTScore-F1']:>8.2f}\n")
        f.write(f"  CIDEr            : {results['CIDEr']:>8.2f}\n")
        f.write("=" * 60 + "\n\n")

        # ── predictions & references ──────────────────────────────────────────
        f.write("  PREDICTIONS & REFERENCES\n")
        f.write("=" * 60 + "\n")
        for vid, pred, refs in zip(video_ids, predictions, references):
            f.write(f"video_id   : {vid}\n")
            f.write(f"prediction : {pred}\n")
            for i, ref in enumerate(refs, 1):
                f.write(f"reference {i} : {ref}\n")
            f.write("-" * 60 + "\n")

    print(f"\n✅ Saved to {OUTPUT_DIR}/results.txt")

In [ ]:
main()